# CrewAI Comprehensive Tutorial Notebook

This notebook is a **combined tutorial** covering three core CrewAI concepts in a single session:

1. **Recipe Recommendation** — Single-agent stateful tasks with `context` chaining
2. **Marketing Research** — Multi-agent collaboration with web search tools
3. **Game Flow** — Event-driven workflows using CrewAI's `Flow` system

> **Note:** Individual, focused notebooks for each topic are also available in this folder. This notebook serves as a unified playground for experimenting with all examples together.

---

In [ ]:
# pip install crewai==0.80.0

# Quick version check to confirm installation
import crewai
print(f"CrewAI version: {crewai.__version__}")

In [ ]:
# Load environment variables (OPENAI_API_KEY, SERPER_API_KEY, etc.)
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from crewai import Agent, Task, Crew, LLM
import os

# Initialize the LLM — used across all examples in this notebook
# llm = LLM(model="gpt-4o-mini")

llm = LLM(
    model="openai/gpt-oss-20b",
    provider="openai",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0,
)

In [ ]:
# (Merged into cell above)
# llm = LLM(model="gpt-4o-mini")

---

# Part 1: Recipe Recommendation (Single Agent, Stateful Tasks)

In this section, we create a **single agent** (Culinary Assistant) that handles two **chained tasks**:
1. Find a recipe matching the user's ingredient and dietary restriction
2. Provide step-by-step cooking instructions for the selected recipe

The key concept here is **task context** — the second task receives the first task's output via the `context` parameter, making the workflow stateful.

In [ ]:
main_ingredient = "tomato"          # Primary ingredient the recipe must include
dietary_restrictions = "shrimps"    # Dietary constraint (e.g., allergens or preferences)

In [ ]:
culinary_assistant = Agent(
    llm=llm,
    role="Culinary Assistant",
    backstory=(
        "An experienced culinary assistant skilled in finding and tailoring "
        "recipes based on ingredients and dietary needs, and providing clear, "
        "step-by-step cooking instructions"
    ),
    goal="Find recipes, filter them to meet dietary preferences, and guide user through recipe steps.",
    verbose=True,  # Prints the agent's reasoning steps
)

In [ ]:
# Task 1: Find and filter recipes based on user inputs
find_and_filter_recipes = Task(
    description=(
        f"Find recipes that use the ingredient: {main_ingredient} and "
        f"filter them to meet dietary restrictions: {dietary_restrictions}."
    ),
    expected_output=f"One recipe using {main_ingredient} and matching {dietary_restrictions} restrictions.",
    agent=culinary_assistant,
)

# Task 2: Generate step-by-step cooking instructions
guide_recipe_steps = Task(
    description="Provide step-by-step instructions for the selected recipe.",
    expected_output="Step-by-step cooking instructions for the chosen recipe.",
    agent=culinary_assistant,
)


In [ ]:
crew = Crew(
    agents=[culinary_assistant],
    tasks=[find_and_filter_recipes, guide_recipe_steps],
    planning=True,  # Enables automatic execution planning
)

In [ ]:
crew.kickoff()

In [ ]:
crew.usage_metrics

---

# Part 2: Marketing Research (Multi-Agent with Tools)

In this section, we build a **two-agent system**:
- **Market Researcher** — Searches the web using `SerperDevTool` for real-time market intelligence
- **Product Strategist** — Synthesizes research findings into a positioning strategy

This demonstrates how different agents can have **different capabilities** (tools) and collaborate in sequence.

In [ ]:
# Agent, Task, Crew already imported above — only need the new tool
# from crewai import Agent, Task, Crew
from crewai_tools import SerperDevTool  # Web search tool (requires SERPER_API_KEY)

In [ ]:
product_name = "energy drink"                     # The product to research and strategize for
strategist_backstory = "marketing strategy"       # Additional expertise area for the strategist

In [ ]:
# (Merged into cell above)
# strategist_backstory = "marketing strategy"

In [ ]:
# Agent 1: Researcher — equipped with web search capability
market_researcher = Agent(
    role="Market Researcher",
    goal="Analyze market trends for the product launch",
    backstory="Experienced in market trends and consumer behavior analysis",
    tools=[SerperDevTool()],  # Gives the agent live web search ability
    verbose=True,
)

# Agent 2: Strategist — synthesizes research into actionable strategy
strategist = Agent(
    role="Product Strategist",
    goal="Create effective positioning strategies for the product",
    backstory=f"Skilled in competitive positioning and {strategist_backstory}",
    verbose=True,
)

In [ ]:
# Task 1: Researcher searches the web for market intelligence
gather_market_insights_task = Task(
    description=(
        f"Browse the internet to gather insights on current market trends "
        f"for the launch of the {product_name} product."
    ),
    expected_output=f"List of relevant market trends and consumer preferences, relevant to {product_name}",
    agent=market_researcher,
)

# Task 2: Strategist builds on the researcher's findings
develop_positioning_strategy_task = Task(
    description=(
        f"Based on the market insights, create a positioning strategy for "
        f"the {product_name} product, including analysis for impact and target audience."
    ),
    expected_output="A positioning strategy with target audience and impact notes",
    agent=strategist,
)

In [ ]:
crew = Crew(
    agents=[market_researcher, strategist],
    tasks=[gather_market_insights_task, develop_positioning_strategy_task],
    planning=True,  # Auto-generates an execution plan before running
)

In [ ]:
output = crew.kickoff()

In [ ]:
print(output.raw)

In [ ]:
output.token_usage

---

# Part 3: Game Flow (Event-Driven Workflows)

This section demonstrates CrewAI's **Flow** system — a way to build structured, deterministic workflows using Python decorators:
- **`@start`** — Marks the entry point
- **`@router`** — Branches execution based on a returned label
- **`@listen`** — Reacts to a specific routing outcome

Unlike Crews (which are LLM-driven), Flows execute pure Python logic with typed state management via Pydantic.

In [ ]:
# Required in Jupyter — allows nested event loops for CrewAI Flows
import nest_asyncio
nest_asyncio.apply()

In [ ]:
import random
from crewai.flow.flow import Flow, listen, router, start  # Flow orchestration decorators
from pydantic import BaseModel  # For typed state management

In [ ]:
# State model — shared across all flow steps via self.state
class GameSession(BaseModel):
    player_won: bool = False


class GameFlow(Flow[GameSession]):
    """A simple flow that simulates a game with win/lose branching."""

    @start()  # Entry point — runs first when flow.kickoff() is called
    def begin_start(self):
        print("Starting the game session")
        player_outcome = random.choice([True, False])
        self.state.player_won = player_outcome  # Write to shared state

    @router(begin_start)
    def check_outcome(self):
        if self.state.player_won:
            return "win"
        else:
            return "lose"

    @listen("win")
    def celebrate_win(self):
        print("Congratulations!")

    @listen("lose")
    def console_loss(self):
        print("Game Over!")

In [ ]:
flow = GameFlow()
result = flow.kickoff()  # Triggers: begin_start → check_outcome → celebrate_win OR console_loss

In [ ]:
# (Merged into cell above)
# flow.kickoff()